# OpenPlaque — Series 6 Native Ostium Topology v1

This experiment stays in **native Series 6 (BestSyst 32%) geometry**.

Primary question: how many distinct CAS-Net coronary/aorta contact clusters exist in Series 6 itself?

The cached Series-6 CAS-Net prediction is reused. A new aorta mask is generated directly from native Series 6 using TotalSegmentator. Series-7 anatomy is used only afterward to label the RCA neighborhood and for descriptive local association; it does not warp the Series-6 coronary tree for the primary contact count.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse/cache controls — immediately after Drive mount
REUSE_NATIVE_AORTA = True
FORCE_NATIVE_AORTA = False

DRIVE_ROOT = "/content/drive/MyDrive/OpenPlaque"
DICOM_ROOT = "/content/drive/MyDrive/CCTA/DICOM/3221"
OUT = f"{DRIVE_ROOT}/Series6_Native_Ostium_Topology_v1"
LOCAL_WORKDIR = "/content/openplaque_series6_native"
CACHED_SERIES6_PREDICTION = f"{DRIVE_ROOT}/Series6_Left_Coronary_Origin_Validation_v1/external_model_predictions/series6_alternate.nii.gz"
NATIVE_AORTA = f"{OUT}/native_aorta/series6_aorta.nii.gz"

print("Output:", OUT)
print("Cached Series 6 CAS-Net:", CACHED_SERIES6_PREDICTION)
print("Native aorta cache:", NATIVE_AORTA)

In [ ]:
import os, shutil, sys, subprocess, json
from pathlib import Path

OPENPLAQUE_PIN = "26595589a3e1cf4f9eacd81411f56750eed7d002"
OPENPLAQUE_BRANCH = "series6-native-ostium-topology-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q pylibjpeg pylibjpeg-libjpeg
%pip install -q /content/OpenPlaque
%pip install -q TotalSegmentator

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

print("OpenPlaque pin:", subprocess.check_output(["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True).strip())
import importlib.metadata
print("TotalSegmentator:", importlib.metadata.version("TotalSegmentator"))


In [ ]:
# Synthetic tests before science
from openplaque.series6_native_ostium_topology_v1 import synthetic_self_test
print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q tests/test_series6_native_ostium_topology_v1.py

In [ ]:
from openplaque.series6_native_ostium_topology_v1 import prepare
from IPython.display import display
import pandas as pd

if not Path(CACHED_SERIES6_PREDICTION).is_file():
    raise FileNotFoundError(
        "Cached Series 6 CAS-Net prediction is missing. Expected: "
        + CACHED_SERIES6_PREDICTION
    )

prep = prepare(
    drive_root=DRIVE_ROOT,
    dicom_root=DICOM_ROOT,
    local_workdir=LOCAL_WORKDIR,
    output_dir=OUT,
)

print("Series 6:", prep["series6_metadata"])
print("Source validation:", prep["series6_brightness"])
print("Root-region translation (labeling only):")
print(json.dumps(prep["root_registration"], indent=2))
print("Native expected RCA root:", prep["series6_expected_rca_root_lps_mm"])
print("Local native Series 6 image:", prep["series6_native_image"])
print("TotalSegmentator DICOM input:", prep["series6_dicom_folder"])


## Native Series-6 aorta

The aorta segmentation is generated from Series 6 itself. The open TotalSegmentator `total` task is restricted to the `aorta` ROI. The resulting mask is cached in this experiment's Drive folder.

In [ ]:
import torch

aorta_cache = Path(NATIVE_AORTA)
aorta_cache.parent.mkdir(parents=True, exist_ok=True)

if REUSE_NATIVE_AORTA and aorta_cache.is_file() and aorta_cache.stat().st_size > 0 and not FORCE_NATIVE_AORTA:
    print("Reusing native Series 6 aorta:", aorta_cache)
else:
    local_out = Path("/content/totalseg_series6_native")
    if local_out.exists():
        shutil.rmtree(local_out)
    local_out.mkdir(parents=True, exist_ok=True)

    cmd = [
        "TotalSegmentator",
        "-i", prep["series6_dicom_folder"],
        "-o", str(local_out),
        "-ta", "total",
        "--roi_subset", "aorta",
        "--nr_thr_saving", "1",
    ]
    print("Running native Series 6 aorta segmentation...")
    print("Command:", " ".join(cmd))
    print("CUDA available:", torch.cuda.is_available())

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"TotalSegmentator failed with exit code {rc}")

    produced = local_out/"aorta.nii.gz"
    if not produced.is_file() or produced.stat().st_size == 0:
        raise FileNotFoundError(f"Expected native aorta output missing: {produced}")
    shutil.copy2(produced, aorta_cache)

print("Native aorta ready:", aorta_cache, aorta_cache.stat().st_size, "bytes")

In [ ]:
from openplaque.series6_native_ostium_topology_v1 import analyze

summary = analyze(
    native_aorta_path=str(aorta_cache),
    cached_prediction_path=CACHED_SERIES6_PREDICTION,
    drive_root=DRIVE_ROOT,
    local_workdir=LOCAL_WORKDIR,
    output_dir=OUT,
)

print(json.dumps(summary["decision"], indent=2))
print("\nStatus:", summary["status"])

In [ ]:
# Quantitative review
display(pd.read_csv(Path(OUT)/"native_aortic_contact_clusters.csv"))
display(pd.read_csv(Path(OUT)/"native_second_contact_candidates.csv"))
display(pd.read_csv(Path(OUT)/"native_component_local_association.csv"))
display(pd.read_csv(Path(OUT)/"gates.csv"))

In [ ]:
# Automatic native contact QC
from IPython.display import Image, display, HTML

display(Image(filename=str(Path(OUT)/"01_native_contact_summary.png"), width=850))
for p in sorted(Path(OUT).glob("QC_contact_*_native_planes.png")):
    print(p.name)
    display(Image(filename=str(p), width=1000))

report = Path(OUT)/"OPENPLAQUE_SERIES6_NATIVE_OSTIUM_TOPOLOGY_V1_REPORT.html"
display(HTML(report.read_text()))

In [ ]:
# Verify deliverables
expected = [
    "run_state.json",
    "summary.json",
    "decision.json",
    "preparation.json",
    "input_provenance.json",
    "native_aortic_contact_clusters.csv",
    "native_second_contact_candidates.csv",
    "native_component_local_association.csv",
    "gates.csv",
    "01_native_contact_summary.png",
    "OPENPLAQUE_SERIES6_NATIVE_OSTIUM_TOPOLOGY_V1_REPORT.html",
    "OPENPLAQUE_SERIES6_NATIVE_OSTIUM_TOPOLOGY_V1_RESULTS.zip",
]
missing=[x for x in expected if not (Path(OUT)/x).exists()]
if missing:
    raise RuntimeError("Missing outputs: "+str(missing))
if not Path(NATIVE_AORTA).is_file():
    raise RuntimeError("Native aorta cache is missing")
print("COMPLETE")
for x in expected:
    print(Path(OUT)/x)